## Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install Dependencies


In [ ]:
!pip install -q ultralytics


## Train YOLO Models


In [ ]:
from ultralytics import YOLO

data_path = "/content/shrimp-yolo/data.yaml"
project_dir = "/content/drive/MyDrive/shrimp/runs"

models = ["yolov8n.pt", "yolov8s.pt", "yolo11n.pt", "yolo11s.pt"]
seeds = [0, 1, 2]

for m in models:
    base_name = m.replace(".pt", "")
    for seed in seeds:
        run_name = f"{base_name}_seed{seed}"
        print(f"\n========== TRAIN {m} | seed={seed} | run={run_name} ==========\n")

        model = YOLO(m)
        model.train(
            data=data_path,
            epochs=150,
            imgsz=512,
            batch=32,
            patience=20,
            project=project_dir,
            name=run_name,
            device=0,
            seed=seed,
            deterministic=True
        )


## Evaluate All Runs On Test Split


In [ ]:
from ultralytics import YOLO
import pandas as pd
import os

data_path = "/content/shrimp-yolo/data.yaml"
project_dir = "/content/drive/MyDrive/shrimp/runs"

models = ["yolov8n", "yolov8s", "yolo11n", "yolo11s"]
seeds = [0, 1, 2]

rows = []

for name in models:
    for seed in seeds:
        run_name = f"{name}_seed{seed}"
        print(f"\n========== EVAL {run_name} (test) ==========\n")

        model_path = f"{project_dir}/{run_name}/weights/best.pt"
        if not os.path.exists(model_path):
            print(f"[WARN] Missing: {model_path}  -> skip")
            continue

        model = YOLO(model_path)

        metrics = model.val(
            data=data_path,
            split="test",
            imgsz=512,
            batch=32,
            device=0,
            verbose=False
        )

        row = {
            "model": name,
            "seed": seed,
            "run": run_name,
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
            "preprocess_ms": float(metrics.speed.get("preprocess", 0.0)),
            "inference_ms": float(metrics.speed.get("inference", 0.0)),
            "postprocess_ms": float(metrics.speed.get("postprocess", 0.0)),
        }
        rows.append(row)

# ----- Per-run results -----
df_runs = pd.DataFrame(rows)
print("\n=== Per-run results ===")
print(df_runs)

save_runs = "/content/drive/MyDrive/shrimp/test_summary_per_run.csv"
df_runs.to_csv(save_runs, index=False)
print(f"\nPer-run results saved to: {save_runs}")

# ----- Aggregate mean ± std by model -----
metric_cols = ["precision", "recall", "mAP50", "mAP50-95", "preprocess_ms", "inference_ms", "postprocess_ms"]

df_summary = (
    df_runs
    .groupby("model")[metric_cols]
    .agg(["mean", "std"])
)

# Flatten columns: precision_mean, precision_std, ...
df_summary.columns = [f"{col}_{stat}" for col, stat in df_summary.columns]
df_summary = df_summary.reset_index()

print("\n=== Summary (mean ± std over seeds) ===")
print(df_summary)

save_summary = "/content/drive/MyDrive/shrimp/test_summary_mean_std.csv"
df_summary.to_csv(save_summary, index=False)
print(f"\nSummary saved to: {save_summary}")


## Evaluate One Best Model


In [ ]:
from ultralytics import YOLO
import pandas as pd
model_name="yolo11n"
data_path = "/content/shrimp-yolo/data.yaml"
project_dir = "/content/drive/MyDrive/shrimp/runs"

model_path = "/content/drive/MyDrive/shrimp/runs/yolo11n_seed0/weights/best.pt"

print(f"\n========== EVAL {model_name} (test) ==========\n")

model = YOLO(model_path)

metrics = model.val(
    data=data_path,
    split="test",
    imgsz=512,
    batch=32,
    device=0,
    verbose=False
)

row = {
    "model": model_name,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
    "preprocess_ms": metrics.speed["preprocess"],
    "inference_ms": metrics.speed["inference"],
    "postprocess_ms": metrics.speed["postprocess"],
}

df = pd.DataFrame([row])
print(df)

save_path = "/content/drive/MyDrive/shrimp/yolov11n_test_summary.csv"
df.to_csv(save_path, index=False)

print(f"\nResults saved to: {save_path}")


## Run Sample Inference


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("/content/drive/MyDrive/shrimp/runs/yolo11n_seed0/weights/best.pt")

# Ghi đè nhãn trong mô hình con
model.model.names = {0: "Healthy", 1: "Diseased"}

test_images = sorted(Path("/content/shrimp-yolo/test/images").glob("*"))
image_path = str(test_images[0])

results = model(
    source=image_path,
    imgsz=512,
    conf=0.25,
    device=0,
    save=False
)

results[0].names = model.model.names

annotated_img = results[0].plot()

plt.figure(figsize=(8, 8))
plt.imshow(annotated_img)
plt.axis("off")
plt.show()

print("Detections:")
for box in results[0].boxes:
    cls_id = int(box.cls)
    conf = float(box.conf)
    print(f"Class: {results[0].names[cls_id]}, Confidence: {conf:.3f}")
